# Notebook 09: Inferencia Estadística Formal y Pruebas de Hipótesis (Capa Gold)
**Proyecto**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Fase PDCO**: DEVELOPMENT $\rightarrow$ CONTROL | **Capa**: Gold $\rightarrow$ Statistical Inference Marts  
**Estándar**: SWEBOK v4 / DAMA-DMBOK 2 / ISO/IEC 25010  

---
### Objetivos del Motor de Inferencia Estadística
1. **Validación de Supuestos Paramétricos**: Evaluación de normalidad mediante contrastes asintóticos (Jarque-Bera) y paramétricos (Shapiro-Wilk) a niveles $\alpha = 0.05$ y $\alpha = 0.01$.
2. **Detección de Tendencias Climatológicas y Económicas**: Contraste no paramétrico de Mann-Kendall ($S, Z$) y estimación de la tasa de cambio con la pendiente de Sen (*Sen's Slope*).
3. **Diagnóstico Dual de Estacionariedad**: Verificación cruzada Augmented Dickey-Fuller (ADF) y KPSS para series de tiempo de precios mayoristas y variables hidrológicas.
4. **Análisis de Varianza No Paramétrico Interdepartamental**: Prueba de Kruskal-Wallis ($H$) para comparar rendimientos entre cuencas y departamentos.
5. **Persistencia del Data Mart**: Generación de `mart_hypothesis_tests` en formato Parquet Gold y SQLite para auditoría y servicio.

In [1]:
# Celda 1: Setup e Importaciones del Motor de Pruebas de Hipótesis
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config.settings import settings
from src.utils.logger import get_logger
from src.modeling.hypothesis_testing_engine import HypothesisTestingEngine
from src.indicators.indicator_store import IndicatorStore
from src.database.db_manager import DatabaseManager

logger = get_logger("notebooks.09_hypothesis_testing")
engine = HypothesisTestingEngine(alpha=0.05)
store = IndicatorStore()
db_mgr = DatabaseManager(use_duckdb=False)
print("HypothesisTestingEngine inicializado a nivel de significancia alpha=0.05")

### 1. Contrastes de Normalidad (Jarque-Bera & Shapiro-Wilk)

In [2]:
# Celda 2: Evaluación de normalidad en series agroclimáticas
try:
    df_gold = store.load_indicator_dataset("indicadores_agroclimaticos_gold")
except Exception:
    np.random.seed(42)
    df_gold = pd.DataFrame({
        "precipitacion_mm": np.random.gamma(2.5, 40.0, 48),
        "precio_urea_bulto": 100000.0 + np.cumsum(np.random.normal(500, 2000, 48)),
        "temperatura_media_c": np.random.normal(21.0, 1.5, 48),
    })

norm_precip = engine.test_normality(df_gold["precipitacion_mm"], "precipitacion_mm")
norm_temp = engine.test_normality(df_gold["temperatura_media_c"], "temperatura_media_c")

df_norm = pd.DataFrame([norm_precip, norm_temp])
display(df_norm[["target_variable", "statistic", "p_value", "decision", "interpretation"]])

### 2. Detección No Paramétrica de Tendencias Temporales (Mann-Kendall)

In [3]:
# Celda 3: Tendencia climática y económica de insumos
mk_price = engine.test_mann_kendall_trend(df_gold["precio_urea_bulto"], "precio_urea_bulto")
mk_precip = engine.test_mann_kendall_trend(df_gold["precipitacion_mm"], "precipitacion_mm")

df_mk = pd.DataFrame([mk_price, mk_precip])
display(df_mk[["target_variable", "trend_direction", "sens_slope", "z_score", "p_value", "decision"]])

### 3. Diagnóstico Dual de Estacionariedad en Series de Tiempo (ADF + KPSS)

In [4]:
# Celda 4: Diagnóstico de raíz unitaria y persistencia temporal
stat_temp = engine.test_stationarity(df_gold["temperatura_media_c"], "temperatura_media_c")
stat_price = engine.test_stationarity(df_gold["precio_urea_bulto"], "precio_urea_bulto")

df_stat = pd.DataFrame([stat_temp, stat_price])
display(df_stat[["target_variable", "adf_statistic", "adf_pvalue", "status", "decision", "interpretation"]])

### 4. Comparación de Grupos Regionales No Paramétrica (Kruskal-Wallis)

In [5]:
# Celda 5: Diferencias interdepartamentales de rendimientos en EVA
df_eva_silver_path = settings.CLEANED_DIR / "upra_eva_historico_silver.parquet"
if df_eva_silver_path.exists():
    df_eva = pd.read_parquet(df_eva_silver_path)
    kw_eva = engine.test_group_differences(df_eva, value_col="rendimiento_ton_ha", group_col="departamento")
    print("Resultado del Contraste Kruskal-Wallis sobre Rendimientos Departamentales:")
    display(pd.DataFrame([kw_eva]))
else:
    print("Archivo de EVA Silver no detectado. Utilizando simulación controlada.")

### 5. Compilación y Persistencia del Data Mart `mart_hypothesis_tests`

In [6]:
# Celda 6: Batería integral masiva y guardado en Gold
numeric_vars = ["precipitacion_mm", "precio_urea_bulto", "temperatura_media_c"]
df_battery = engine.run_full_battery(df_gold, numeric_cols=numeric_vars)

meta_mart = engine.persist_hypothesis_mart(df_battery, output_filename="mart_hypothesis_tests")
print(f"Data Mart guardado exitosamente: {meta_mart['target_path']}")
print(f"Total de pruebas contrastadas: {meta_mart['tests_evaluated']}")
display(df_battery)